In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
np.random.seed(42)

n = 2000

companies = [
    "Pfizer", "Roche", "Moderna", "Novartis",
    "AstraZeneca", "Merck", "GSK",
    "Johnson & Johnson", "Sanofi", "Bayer"
]

drugs = [
    "OncoX", "CardioPlus", "NeuroVax", "ImmunoGen",
    "Respira", "MetaCure", "PainFree", "BioStat"
]

trial_phases = [
    "Phase I", "Phase II", "Phase III", "Approved"
]

regions = [
    "North America", "Europe", "Asia",
    "South America", "Middle East"
]

In [ ]:
df = pd.DataFrame({
    "drug_name": np.random.choice(drugs, n),
    "trial_phase": np.random.choice(
        trial_phases,
        n,
        p=[0.25, 0.30, 0.30, 0.15]
    ),
    "success_rate": np.random.normal(75, 10, n).clip(40, 99),
    "patients_enrolled": np.random.randint(100, 10000, n),
    "side_effect_rate": np.random.normal(12, 5, n).clip(1, 40),
    "company": np.random.choice(companies, n),
    "region": np.random.choice(regions, n),
    "trial_cost_million_usd": np.random.randint(5, 500, n),
    "approval_probability": np.random.uniform(0.2, 0.98, n)
})

df.head()

In [ ]:
df["risk_score"] = (
    df["side_effect_rate"] / df["success_rate"]
)

df["risk_category"] = pd.cut(
    df["risk_score"],
    bins=[0, 0.1, 0.2, 1],
    labels=["Low", "Medium", "High"]
)

df["successful_trial"] = (
    df["success_rate"] > 75
).astype(int)

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
top_companies = (
    df.groupby("company")["success_rate"]
    .mean()
    .sort_values(ascending=False)
)

top_companies.plot(kind="bar")

plt.title("Average Success Rate by Company")
plt.xlabel("Company")
plt.ylabel("Success Rate")

plt.show()

In [ ]:
risk_phase = (
    df.groupby("trial_phase")["risk_score"]
    .mean()
)

risk_phase.plot(kind="line", marker="o")

plt.title("Risk Score by Trial Phase")
plt.xlabel("Trial Phase")
plt.ylabel("Risk Score")

plt.show()

In [ ]:
corr = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(10,7))

sns.heatmap(corr, annot=True)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

features = [
    "patients_enrolled",
    "side_effect_rate",
    "trial_cost_million_usd",
    "approval_probability"
]

X = df[features]
y = df["successful_trial"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Model Accuracy:", accuracy)

In [ ]:
df.to_csv("cleaned_pharma_data_pro.csv", index=False)

print("Dataset exported successfully")

In [ ]:
from google.colab import files

files.download("cleaned_pharma_data_pro.csv")